<a href="https://colab.research.google.com/github/Radhakuchekar/Preparation/blob/pyspark/SQL_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
from pyspark.sql.functions import *
spark

In [ ]:
from pyspark.sql.functions import *

In [ ]:
cols = ['trade_id','customer_id','trade_date','trade_amount']
data= [[1001,123,'06/10/2022',10000],
[1002,456,'06/11/2022',20000],
[1003,789,'06/12/2022',30000],
[1004,123,'07/10/2022',10000],
[1005,123,'07/10/2022',10005],
[1005,123,'06/10/2022',10005],
[1005,456,'07/12/2022',15000],
[1006,789,'07/12/2022',35000],
[1007,123,'07/15/2022',30000]]
df = spark.createDataFrame(data, cols)
df.createOrReplaceTempView("trades")

In [ ]:
df.show()

+--------+-----------+----------+------------+
|trade_id|customer_id|trade_date|trade_amount|
+--------+-----------+----------+------------+
|    1001|        123|06/10/2022|       10000|
|    1002|        456|06/11/2022|       20000|
|    1003|        789|06/12/2022|       30000|
|    1004|        123|07/10/2022|       10000|
|    1005|        123|07/10/2022|       10005|
|    1005|        123|06/10/2022|       10005|
|    1005|        456|07/12/2022|       15000|
|    1006|        789|07/12/2022|       35000|
|    1007|        123|07/15/2022|       30000|
+--------+-----------+----------+------------+



In [ ]:
from pyspark.sql.functions import col

In [ ]:
spark.sql("""SELECT
    date_format(to_date(trade_date, 'M/d/yyyy'), 'M') AS month,
    customer_id,
    SUM(trade_amount) AS total_amount,
    rank() over (partition by date_format(to_date(trade_date, 'M/d/yyyy'), 'M') order by SUM(trade_amount) desc) as rank_amount
FROM
    trades
group by
    date_format(to_date(trade_date, 'M/d/yyyy'), 'M'),
    customer_id
order by
    month,
    rank_amount
""").filter(col("rank_amount")<=52).drop('rank_amount').show()

+-----+-----------+------------+
|month|customer_id|total_amount|
+-----+-----------+------------+
|    6|        789|       30000|
|    6|        123|       20005|
|    6|        456|       20000|
|    7|        123|       50005|
|    7|        789|       35000|
|    7|        456|       15000|
+-----+-----------+------------+



In [ ]:
spark.sql("select now() as date").show(truncate=False)
spark.sql("select current_date() as date").show(truncate=False)

+----------------------+
|date                  |
+----------------------+
|2025-01-19 02:47:20.81|
+----------------------+

+----------+
|date      |
+----------+
|2025-01-19|
+----------+



In [ ]:
spark.sql("select trade_date,date_format(to_date(trade_date, 'M/d/yyyy'), 'M') as month from trades").show(truncate=False)

+----------+-----+
|trade_date|month|
+----------+-----+
|06/10/2022|6    |
|06/11/2022|6    |
|06/12/2022|6    |
|07/10/2022|7    |
|07/12/2022|7    |
|07/12/2022|7    |
|07/15/2022|7    |
+----------+-----+



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Define the schema for the DataFrame
schema = StructType([
    StructField("trade_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("trade_date", StringType(), True),  # Trade date as String
    StructField("trade_amount", IntegerType(), True)
])

# Define the data for the DataFrame
data = [
    (1001, 123, '06/10/2022', 10000),
    (1002, 456, '06/11/2022', 20000),
    (1003, 789, '06/12/2022', 30000),
    (1004, 123, '07/10/2022', 10000),
    (1005, 123, '07/10/2022', 10005),
    (1005, 123, '06/10/2022', 10005),
    (1005, 456, '07/12/2022', 15000),
    (1006, 789, '07/12/2022', 35000),
    (1007, 123, '07/15/2022', 30000)
]

# Create the DataFrame
trades_df = spark.createDataFrame(data, schema)

# Show the DataFrame
trades_df.show()


+--------+-----------+----------+------------+
|trade_id|customer_id|trade_date|trade_amount|
+--------+-----------+----------+------------+
|    1001|        123|06/10/2022|       10000|
|    1002|        456|06/11/2022|       20000|
|    1003|        789|06/12/2022|       30000|
|    1004|        123|07/10/2022|       10000|
|    1005|        123|07/10/2022|       10005|
|    1005|        123|06/10/2022|       10005|
|    1005|        456|07/12/2022|       15000|
|    1006|        789|07/12/2022|       35000|
|    1007|        123|07/15/2022|       30000|
+--------+-----------+----------+------------+



In [ ]:
from pyspark.sql.functions import *
with_year = trades_df.withColumn("year", date_format(to_date("trade_date", "M/d/yyyy"), "yyyy"))

In [ ]:
with_year.show()

+--------+-----------+----------+------------+----+
|trade_id|customer_id|trade_date|trade_amount|year|
+--------+-----------+----------+------------+----+
|    1001|        123|06/10/2022|       10000|2022|
|    1002|        456|06/11/2022|       20000|2022|
|    1003|        789|06/12/2022|       30000|2022|
|    1004|        123|07/10/2022|       10000|2022|
|    1005|        123|07/10/2022|       10005|2022|
|    1005|        123|06/10/2022|       10005|2022|
|    1005|        456|07/12/2022|       15000|2022|
|    1006|        789|07/12/2022|       35000|2022|
|    1007|        123|07/15/2022|       30000|2022|
+--------+-----------+----------+------------+----+



In [ ]:
max_sal = with_year.groupBy(["year", "customer_id"]).agg(max("trade_amount").alias("trade_amount"))

In [ ]:
result = with_year.join(max_sal, ['year','customer_id','trade_amount'], "inner")

In [ ]:
result.show()

+----+-----------+------------+--------+----------+
|year|customer_id|trade_amount|trade_id|trade_date|
+----+-----------+------------+--------+----------+
|2022|        456|       20000|    1002|06/11/2022|
|2022|        123|       30000|    1007|07/15/2022|
|2022|        789|       35000|    1006|07/12/2022|
+----+-----------+------------+--------+----------+



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType
from pyspark.sql.functions import *

# Define the schema for the DataFrame
schema = StructType([
    StructField("trade_id", IntegerType(), True),       # trade_id as Integer
    StructField("trader_id", IntegerType(), True),      # trader_id as Integer
    StructField("trade_amount", IntegerType(), True),  # trade_amount as Decimal (15 digits, 2 decimals)
    StructField("trade_date", StringType(), True)       # trade_date as String
])

# Define the data for the DataFrame
data = [
    (1001, 12501, 100000, '2018-06-12'),
    (1002, 12501, 200000, '2018-07-15'),
    (1010, 12502, 50000, '2018-06-18'),
    (1011, 12502, 45000, '2018-08-20'),
    (1012, 12501, 250000, '2019-06-15'),
    (1013, 12502, 80000, '2019-05-10')
]

# Create the DataFrame
transactions_df = spark.createDataFrame(data, schema)

# Show the DataFrame
transactions_df.show()


+--------+---------+------------+----------+
|trade_id|trader_id|trade_amount|trade_date|
+--------+---------+------------+----------+
|    1001|    12501|      100000|2018-06-12|
|    1002|    12501|      200000|2018-07-15|
|    1010|    12502|       50000|2018-06-18|
|    1011|    12502|       45000|2018-08-20|
|    1012|    12501|      250000|2019-06-15|
|    1013|    12502|       80000|2019-05-10|
+--------+---------+------------+----------+



In [ ]:
transactions_df = transactions_df.withColumn("year", date_format(to_date('trade_date', 'yyyy-d-m'), 'yyyy'))

In [ ]:
transactions_df.show()

+--------+---------+------------+----------+----+
|trade_id|trader_id|trade_amount|trade_date|year|
+--------+---------+------------+----------+----+
|    1001|    12501|      100000|2018-06-12|2018|
|    1002|    12501|      200000|2018-07-15|2018|
|    1010|    12502|       50000|2018-06-18|2018|
|    1011|    12502|       45000|2018-08-20|2018|
|    1012|    12501|      250000|2019-06-15|2019|
|    1013|    12502|       80000|2019-05-10|2019|
+--------+---------+------------+----------+----+



In [ ]:
from pyspark.sql.window import Window
windowSpec = Window.partitionBy(["trader_id","year"]).orderBy(desc("trade_amount"))

In [ ]:
transactions_df_with_rank = transactions_df.withColumn("rank", rank().over(windowSpec))
transactions_df_with_rank.filter("rank = 1").select(transactions_df['*']).show()

+--------+---------+------------+----------+----+
|trade_id|trader_id|trade_amount|trade_date|year|
+--------+---------+------------+----------+----+
|    1002|    12501|      200000|2018-07-15|2018|
|    1012|    12501|      250000|2019-06-15|2019|
|    1010|    12502|       50000|2018-06-18|2018|
|    1013|    12502|       80000|2019-05-10|2019|
+--------+---------+------------+----------+----+



In [ ]:
# trade_desc, trade_date, total volume
trades = [
    [1821, 1, "2022-07-01", 43000],
    [4935, 2, "2022-07-01", 38500],
    [4935, 2, "2022-07-01", 3950000],
    [3492, 1, "2022-07-02", 25000],
    [3753, 2, "2022-07-02", 28000],
    [2835, 1, "2022-07-03", 60000]
]

# Trade types
trade_types = [
    [1, "Buy"],
    [2, "Sell"]
]

In [ ]:
trades_cols = ['trade_id','trade_type_id',	'trade_date',	'volume']
trade_types_cols  =['trade_type_id','trade_desc']
trades = spark.createDataFrame(trades, trades_cols)
trade_types = spark.createDataFrame(trade_types, trade_types_cols)

In [ ]:
trades.createOrReplaceTempView("trades")
trade_types.createOrReplaceTempView("trade_types")



In [ ]:
joined_trades_df = trades.join(trade_types, on="trade_type_id", how="inner")
joined_trades_df = joined_trades_df.withColumn("volume", col('volume').cast("int"))

In [ ]:
result_df = (
    joined_trades_df
    .groupBy("trade_desc", "trade_date")
    .agg(sum("volume").alias("total_volume"))
)

# Show the results
result_df.show(truncate=False)

+----------+----------+------------+
|trade_desc|trade_date|total_volume|
+----------+----------+------------+
|Sell      |2022-07-02|28000       |
|Sell      |2022-07-01|3988500     |
|Buy       |2022-07-02|25000       |
|Buy       |2022-07-01|43000       |
|Buy       |2022-07-03|60000       |
+----------+----------+------------+



In [ ]:
trades.show()

+--------+-------------+----------+-------+
|trade_id|trade_type_id|trade_date| volume|
+--------+-------------+----------+-------+
|    1821|            1|2022-07-01|  43000|
|    4935|            2|2022-07-01|  38500|
|    4935|            2|2022-07-01|3950000|
|    3492|            1|2022-07-02|  25000|
|    3753|            2|2022-07-02|  28000|
|    2835|            1|2022-07-03|  60000|
+--------+-------------+----------+-------+



In [ ]:
spark.sql("select tt.trade_desc, t.trade_date, sum(volume) from trades t join trade_types tt on t.trade_type_id=tt.trade_type_id group by tt.trade_desc, t.trade_date order by  tt.trade_desc  ").show(truncate = False)

+----------+----------+-----------+
|trade_desc|trade_date|sum(volume)|
+----------+----------+-----------+
|Buy       |2022-07-02|25000      |
|Buy       |2022-07-01|43000      |
|Buy       |2022-07-03|60000      |
|Sell      |2022-07-02|28000      |
|Sell      |2022-07-01|78000      |
+----------+----------+-----------+



In [ ]:
# Ad clicks data as a list of lists
ad_clicks_data = [
    [101, 1, "Facebook", "03/01/2022 00:00:00"],
    [102, 2, "Google Ads", "03/01/2022 00:00:00"],
    [103, 1, "Facebook", "03/01/2022 00:00:00"],
    [104, 3, "LinkedIn", "03/02/2022 00:00:00"],
    [105, 1, "Facebook", "03/03/2022 00:00:00"]
]
ad_clicks_columns = ["click_id", "ad_id", "platform", "click_date"]

# Ad impressions data as a list of lists
ad_impressions_data = [
    [201, 1, "Facebook", "03/01/2022 00:00:00"],
    [202, 2, "Google Ads", "03/01/2022 00:00:00"],
    [203, 3, "LinkedIn", "03/01/2022 00:00:00"],
    [204, 1, "Facebook", "03/01/2022 00:00:00"],
    [205, 2, "Google Ads", "03/01/2022 00:00:00"]
]
ad_impressions_columns = ["impression_id", "ad_id", "platform", "impression_date"]

# Create DataFrames
ad_clicks_df = spark.createDataFrame(ad_clicks_data, schema=ad_clicks_columns)
ad_impressions_df = spark.createDataFrame(ad_impressions_data, schema=ad_impressions_columns)

# Show the DataFrames
print("Ad Clicks DataFrame:")
ad_clicks_df.show(truncate=False)

print("Ad Impressions DataFrame:")
ad_impressions_df.show(truncate=False)


Ad Clicks DataFrame:
+--------+-----+----------+-------------------+
|click_id|ad_id|platform  |click_date         |
+--------+-----+----------+-------------------+
|101     |1    |Facebook  |03/01/2022 00:00:00|
|102     |2    |Google Ads|03/01/2022 00:00:00|
|103     |1    |Facebook  |03/01/2022 00:00:00|
|104     |3    |LinkedIn  |03/02/2022 00:00:00|
|105     |1    |Facebook  |03/03/2022 00:00:00|
+--------+-----+----------+-------------------+

Ad Impressions DataFrame:
+-------------+-----+----------+-------------------+
|impression_id|ad_id|platform  |impression_date    |
+-------------+-----+----------+-------------------+
|201          |1    |Facebook  |03/01/2022 00:00:00|
|202          |2    |Google Ads|03/01/2022 00:00:00|
|203          |3    |LinkedIn  |03/01/2022 00:00:00|
|204          |1    |Facebook  |03/01/2022 00:00:00|
|205          |2    |Google Ads|03/01/2022 00:00:00|
+-------------+-----+----------+-------------------+



In [ ]:
# total number of impressions, clicks, and
# the click-through rate (CTR) for each ad on each platform for the month of March.

In [ ]:
ad_impressions_df=ad_impressions_df.filter("date_format(to_date(impression_date, 'dd/MM/yyyy'), 'MM') ==3")
joined_df = ad_clicks_df.join(ad_impressions_df, ['ad_id','platform'], 'inner')

In [ ]:
# total number of impressions, clicks, and
# the click-through rate (CTR) for each ad on each platform for the month of March.

In [ ]:

# Define the data
data = [
    (6759, "John", "Smith", "New York", "New York", 12500.75),
    (1231, "Jennifer", "Lawrence", "Los Angeles", "California", 8750.80),
    (4852, "James", "Franklin", "New York", "New York", 10930.35),
    (8654, "Janet", "King", "San Francisco", "California", 9800.45),
    (2331, "Jessica", "Moore", "Chicago", "Illinois", 10450.60),
    (7893, "Jacob", "Martin", "New York", "New York", 11700.25),
]

# Define the schema
columns = ["customer_id", "first_name", "last_name", "city", "state", "account_balance"]

# Create the DataFrame
customer_df = spark.createDataFrame(data, schema=columns)

# Show the DataFrame
customer_df.show()


+-----------+----------+---------+-------------+----------+---------------+
|customer_id|first_name|last_name|         city|     state|account_balance|
+-----------+----------+---------+-------------+----------+---------------+
|       6759|      John|    Smith|     New York|  New York|       12500.75|
|       1231|  Jennifer| Lawrence|  Los Angeles|California|         8750.8|
|       4852|     James| Franklin|     New York|  New York|       10930.35|
|       8654|     Janet|     King|San Francisco|California|        9800.45|
|       2331|   Jessica|    Moore|      Chicago|  Illinois|        10450.6|
|       7893|     Jacob|   Martin|     New York|  New York|       11700.25|
+-----------+----------+---------+-------------+----------+---------------+



In [ ]:
#  first names start with 'J' and reside in 'New York'

#  select * from
customer_df.createGlobalTempView("customers")

In [ ]:
spark.sql("select * from global_temp.customers where first_name like 'J%' and state = 'New York'").show()

+-----------+----------+---------+--------+--------+---------------+
|customer_id|first_name|last_name|    city|   state|account_balance|
+-----------+----------+---------+--------+--------+---------------+
|       6759|      John|    Smith|New York|New York|       12500.75|
|       4852|     James| Franklin|New York|New York|       10930.35|
|       7893|     Jacob|   Martin|New York|New York|       11700.25|
+-----------+----------+---------+--------+--------+---------------+



In [ ]:
customer_df.filter("first_name like 'J%' and state = 'New York'").show()

+-----------+----------+---------+--------+--------+---------------+
|customer_id|first_name|last_name|    city|   state|account_balance|
+-----------+----------+---------+--------+--------+---------------+
|       6759|      John|    Smith|New York|New York|       12500.75|
|       4852|     James| Franklin|New York|New York|       10930.35|
|       7893|     Jacob|   Martin|New York|New York|       11700.25|
+-----------+----------+---------+--------+--------+---------------+



In [ ]:
# create filter condittion in different ways
customer_df.filter((col("first_name").startswith("J")) & (col("state") == "New York"))
customer_df.filter(col("first_name").startswith("J")).filter(col("state") == "New York")
customer_df.filter(col("first_name").startswith("J"), col("state") == "New York")
customer_df.filter(col("first_name").startswith("J"), col("state") == "New York")


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum

# Initialize Spark session
spark = SparkSession.builder.appName("PivotExample").getOrCreate()

# Input data
data = [
    ("Personal", "North", 200000),
    ("Personal", "South", 150000),
    ("Business", "North", 300000),
    ("Business", "South", 250000)
]

# Define schema and create DataFrame
columns = ["Loan_Type", "Region", "Amount"]
df = spark.createDataFrame(data, columns)

# Show the input DataFrame
df.show()


+---------+------+------+
|Loan_Type|Region|Amount|
+---------+------+------+
| Personal| North|200000|
| Personal| South|150000|
| Business| North|300000|
| Business| South|250000|
+---------+------+------+



In [ ]:
df.createOrReplaceTempView("loan_data")

In [ ]:
spark.sql("select Loan_Type from loan_data").show()

+---------+
|Loan_Type|
+---------+
| Personal|
| Personal|
| Business|
| Business|
+---------+



In [ ]:
df.groupBy("Loan_Type").pivot("Region").sum("Amount").show()

+---------+------+------+
|Loan_Type| North| South|
+---------+------+------+
| Personal|200000|150000|
| Business|300000|250000|
+---------+------+------+



In [ ]:
spark.sql("select Loan_Type, sum(case when Region = 'North' then Amount else 0 end) as North, sum(case when Region = 'South' then Amount else 0 end) as South from loan_data group by Loan_Type").show()

+---------+------+------+
|Loan_Type| North| South|
+---------+------+------+
| Personal|200000|150000|
| Business|300000|250000|
+---------+------+------+



In [ ]:
data = [
    (101, "North", "Q1", 20000),
    (101, "South", "Q1", 15000),
    (102, "North", "Q2", 30000),
    (102, "South", "Q2", 25000)
]
columns = ["product_id", "region", "quarter", "sales_amount"]
sales_df = spark.createDataFrame(data, columns)

In [ ]:
sales_df = sales_df.withColumn("pivotCol", concat(col("region"), lit("-"), col("quarter")))

In [ ]:
sales_df.show()

+----------+------+-------+------------+--------+
|product_id|region|quarter|sales_amount|pivotCol|
+----------+------+-------+------------+--------+
|       101| North|     Q1|       20000|North-Q1|
|       101| South|     Q1|       15000|South-Q1|
|       102| North|     Q2|       30000|North-Q2|
|       102| South|     Q2|       25000|South-Q2|
+----------+------+-------+------------+--------+



In [ ]:
sales_df.groupBy("product_id").pivot("pivotCol").sum('sales_amount')

product_id,North-Q1,North-Q2,South-Q1,South-Q2
101,20000,NULL,15000,NULL
102,NULL,30000,NULL,25000


In [ ]:
data = [
    (101, "Region-qurter", "N-Q1", 100),
    (101, "Region-qurter", "S-Q1", 200),
    (102, "Region-qurter", "N-Q2", 300),
    (102, "Region-qurter", "N-Q2", 400)
]
columns = ["product_id", "seg_key", "seg_val", "value"]
sales_df = spark.createDataFrame(data, columns)

In [ ]:
sales_df = sales_df.withColumn("seg_key", explode(split(sales_df.seg_key, "-")))
sales_df=sales_df.withColumn("seg_val", explode(split(sales_df.seg_val, "-")))
sales_df.show()

+----------+-------+-------+-----+
|product_id|seg_key|seg_val|value|
+----------+-------+-------+-----+
|       101| Region|      N|  100|
|       101| Region|     Q1|  100|
|       101| qurter|      N|  100|
|       101| qurter|     Q1|  100|
|       101| Region|      S|  200|
|       101| Region|     Q1|  200|
|       101| qurter|      S|  200|
|       101| qurter|     Q1|  200|
|       102| Region|      N|  300|
|       102| Region|     Q2|  300|
|       102| qurter|      N|  300|
|       102| qurter|     Q2|  300|
|       102| Region|      N|  400|
|       102| Region|     Q2|  400|
|       102| qurter|      N|  400|
|       102| qurter|     Q2|  400|
+----------+-------+-------+-----+



In [ ]:
sales_df = sales_df.withColumn("seg_key_array", split(col("seg_key"), "-"))
sales_df = sales_df.withColumn("seg_val_array", split(col("seg_val"), "-"))

# Ensure that both arrays have the same length by zipping them together
sales_df = sales_df.withColumn("zipped", array("seg_key_array", "seg_val_array"))
sales_df.show(truncate=False)

# Explode the zipped array to create separate rows for each key-value pair
sales_df = sales_df.withColumn("seg_pair", explode(col("zipped")))
sales_df.show(truncate=False)

# Explode the individual elements of the tuple
sales_df = sales_df.withColumn("seg_key", col("seg_pair")[0]).withColumn("seg_val", col("seg_pair")[1])
sales_df.show(truncate=False)

# Drop the intermediate columns
sales_df = sales_df.drop("seg_key_array", "seg_val_array", "zipped", "seg_pair")


sales_df.show(truncate=False)

+----------+-------+-------+-----+-------------+-------------+--------------------+
|product_id|seg_key|seg_val|value|seg_key_array|seg_val_array|zipped              |
+----------+-------+-------+-----+-------------+-------------+--------------------+
|101       |Region |qurter |100  |[Region]     |[qurter]     |[[Region], [qurter]]|
|101       |N      |Q1     |100  |[N]          |[Q1]         |[[N], [Q1]]         |
|101       |Region |qurter |200  |[Region]     |[qurter]     |[[Region], [qurter]]|
|101       |S      |Q1     |200  |[S]          |[Q1]         |[[S], [Q1]]         |
|102       |Region |qurter |300  |[Region]     |[qurter]     |[[Region], [qurter]]|
|102       |N      |Q2     |300  |[N]          |[Q2]         |[[N], [Q2]]         |
|102       |Region |qurter |400  |[Region]     |[qurter]     |[[Region], [qurter]]|
|102       |N      |Q2     |400  |[N]          |[Q2]         |[[N], [Q2]]         |
+----------+-------+-------+-----+-------------+-------------+--------------

In [ ]:
sales_df = sales_df.withColumn("seg_key", explode(split(sales_df.seg_key, "-")))
sales_df = sales_df.withColumn("seg_val", explode(split(sales_df.seg_val, "-")))

In [ ]:
sales_df.show(truncate=False)

+----------+-------+-------+-----+
|product_id|seg_key|seg_val|value|
+----------+-------+-------+-----+
|101       |Region |N      |100  |
|101       |Region |Q1     |100  |
|101       |qurter |N      |100  |
|101       |qurter |Q1     |100  |
|101       |Region |S      |200  |
|101       |Region |Q1     |200  |
|101       |qurter |S      |200  |
|101       |qurter |Q1     |200  |
|102       |Region |N      |300  |
|102       |Region |Q2     |300  |
|102       |qurter |N      |300  |
|102       |qurter |Q2     |300  |
|102       |Region |N      |400  |
|102       |Region |Q2     |400  |
|102       |qurter |N      |400  |
|102       |qurter |Q2     |400  |
+----------+-------+-------+-----+



In [ ]:
sales_df.groupBy("product_id").pivot("seg_key").agg(max("seg_val"))

product_id,Region,qurter
101,S,S
102,Q2,Q2


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.master("local").appName("PivotExample").getOrCreate()

# Sample data
data = [
    (101, "Region-qurter", "N-Q1", 100),
    (101, "Region-qurter", "S-Q1", 200),
    (102, "Region-qurter", "N-Q2", 300),
    (102, "Region-qurter", "N-Q2", 400)
]

# Creating DataFrame
columns = ["product_id", "seg_key", "seg_val", "value"]
sales_df = spark.createDataFrame(data, columns)

# Extract region and quarter from seg_val
sales_df = sales_df.withColumn("region", F.split(F.col("seg_val"), "-").getItem(0)) \
                   .withColumn("quarter", F.split(F.col("seg_val"), "-").getItem(1))

# Select only relevant columns and display the results
result_df = sales_df.select("product_id", "region", "quarter", "value")

# Show the result
result_df.show(truncate=False)


+----------+------+-------+-----+
|product_id|region|quarter|value|
+----------+------+-------+-----+
|101       |N     |Q1     |100  |
|101       |S     |Q1     |200  |
|102       |N     |Q2     |300  |
|102       |N     |Q2     |400  |
+----------+------+-------+-----+



In [ ]:
+----------+-------+-------+-----+
|product_id|region|quarter|value|
+----------+-------+-------+-----+
|101       |N |Q1     |100  |